In [11]:
%cd "E:/src code 2/python 2/KG"
import json
import pandas as pd
import os

from src.index.entity_extractor import EntityExtractor
from src.utils.config_loader import ConfigLoader
from src.llm.gemini import Gemini_LLM
from src.utils.utils import read_file
from src.utils.type import TYPE_OF_ENTITY_IN_KG, TYPE_OF_JOB, TYPE_OF_JOB_ENTITY, TYPE_OF_CV, TYPE_OF_EDGE
from src.db.neo4j import GraphManager, Node, Edge
gm = GraphManager("neo4j://localhost:7687", "neo4j", "123123aA@")
config = ConfigLoader().get_config_from_file(r"E:\src code 2\python 2\Legal_RAG\config\config.yaml")
llm = Gemini_LLM(config=config)

entities = ["Programming Language", "Library", "Software", "Technology", "Task", "Country", "City", "District"]

E:\src code 2\python 2\KG


In [13]:
gm.delete_data()

Đã xóa thành công toàn bộ dữ liệu trong Neo4j.


# Load data

In [14]:
folder = "E:/data/kg_processed/"

for file in ["graph.txt"]:#os.listdir(folder):
    data = read_file(folder + file)
    for line in data.split("\n"):
        if len(line) < 3: continue
        e1, e2 = line.split("\t")
        n1 = Node(TYPE_OF_ENTITY_IN_KG, {"name":e1})
        n2 = Node(TYPE_OF_ENTITY_IN_KG, {"name":e2})
        gm.add_node(n1)
        gm.add_node(n2)
        gm.add_edge(Edge(n1, n2, TYPE_OF_EDGE))
        

In [15]:
with open("E:/data/place/city.txt", 'r', encoding="utf-8") as f:
    data = f.read()
for line in data.split('\n'):
    if len(line) < 3: continue
    _, e2, e1 = line.split("\t")
    n1 = Node(TYPE_OF_ENTITY_IN_KG, {"name":e1})
    n2 = Node(TYPE_OF_ENTITY_IN_KG, {"name":e2})
    gm.add_node(n1)
    gm.add_node(n2)
    gm.add_edge(Edge(n1, n2, TYPE_OF_EDGE))

In [16]:
with open("E:/data/place/district.txt", 'r', encoding="utf-8") as f:
    data = f.read()
for line in data.split('\n'):
    if len(line) < 3: continue
    _, e1, e2s = line.split("\t")
    for e2 in e2s.split(", "):
        n1 = Node(TYPE_OF_ENTITY_IN_KG, {"name":e1})
        n2 = Node(TYPE_OF_ENTITY_IN_KG, {"name":e2})
        gm.add_node(n1)
        gm.add_node(n2)
        gm.add_edge(Edge(n1, n2, TYPE_OF_EDGE))

In [17]:
with open("E:/data/place/region.txt", 'r', encoding="utf-8") as f:
    data = f.read()
for line in data.split('\n'):
    if len(line) < 3: continue
    e1, e2 = line.split("\t")
    n1 = Node(TYPE_OF_ENTITY_IN_KG, {"name":e1})
    n2 = Node(TYPE_OF_ENTITY_IN_KG, {"name":e2})
    gm.add_node(n1)
    gm.add_node(n2)
    gm.add_edge(Edge(n1, n2, TYPE_OF_EDGE))

# Load JD

In [18]:
folder = "E:/data/job/"
entity_extractor = EntityExtractor(entities=", ".join(entities), llm=llm, cache_folder=folder)
for file in os.listdir(folder):
    if not file.endswith("_v3.txt"): continue
    entities_list, _ = entity_extractor.parse_entities(read_file(folder + file))
    job_name = file[:-4]
    job_node = Node(TYPE_OF_JOB, {"name":job_name})
    gm.add_edge(job_node)
    for e in entities_list:
        e = e.node()
        e.label = TYPE_OF_JOB_ENTITY
        gm.add_node(e)
        gm.add_edge(Edge(job_node, e, TYPE_OF_EDGE))

# Load CV

In [19]:
folder = "E:/data/cv/"
entity_extractor = EntityExtractor(entities=", ".join(entities), llm=llm, cache_folder=folder)
for file in os.listdir(folder):
    if "raw" in file: continue
    with open(folder + file, 'r', encoding="utf-8") as f:
        try:
            data = f.read()
            entities_list, _ = entity_extractor.parse_entities(data)
            list_node = [e.node() for e in entities_list]
            cv_node = Node(TYPE_OF_CV, {"name":file[:-4]})
            for e in list_node:
                e.label = TYPE_OF_JOB_ENTITY
                gm.add_node(e)
                gm.add_edge(Edge(cv_node, e, TYPE_OF_EDGE))
        except:
            print(file)

Java Developer_33.txt
Mechanical Engineer_7.txt


In [20]:
list_neighbor = gm.get_relations_to_node(Node(TYPE_OF_ENTITY_IN_KG, {"name": "Hà Nội"}))
for neighbor in list_neighbor:
    neighbor_node = neighbor.node
    print(neighbor_node.properties['name'])

bắc_bộ


In [21]:
gm.jaccard_similarity_top(Node(TYPE_OF_CV, {"name" : "Java Developer_32"}), TYPE_OF_JOB)

[('115_v3', 0.07894736842105263),
 ('165_v3', 0.075),
 ('15_v3', 0.075),
 ('158_v3', 0.075),
 ('258_v3', 0.075)]

In [22]:
list_neighbor = gm.get_relations_to_node(Node(TYPE_OF_JOB_ENTITY, {"name": "python"}))
for neighbor in list_neighbor:
    neighbor_node = neighbor.node
    print(neighbor_node.properties['name'])

50_v3
python_developer_9
python_developer_8
python_developer_7
python_developer_6
python_developer_5
python_developer_4
python_developer_3
python_developer_2
python_developer_15
python_developer_14
python_developer_13
python_developer_12
python_developer_11
python_developer_10
python_developer_1
python_developer_0
hr_19
health_and_fitness_7
health_and_fitness_5
health_and_fitness_1
hadoop_5
hadoop_4
hadoop_3
hadoop_2
hadoop_1
hadoop_0
electrical_engineering_5
electrical_engineering_4
electrical_engineering_3
electrical_engineering_2
electrical_engineering_1
electrical_engineering_0
data_science_9
data_science_8
data_science_7
data_science_6
data_science_5
data_science_4
data_science_3
data_science_2
data_science_15
data_science_14
data_science_13
data_science_12
data_science_11
data_science_10
data_science_1
data_science_0
civil_engineer_2
civil_engineer_0
blockchain_7
blockchain_4
blockchain_22
blockchain_19
blockchain_16
blockchain_13
blockchain_10
blockchain_1
arts_5
arts_4
arts_2
a